# 04 — MongoDB Verification

Confirm `stream_live.py` is storing completed candles into `closed_candles`:
1. Total & per-symbol counts
2. Latest candles
3. Duplicate check (unique index should prevent any)
4. Gap check between consecutive 1-min candles

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))  # so `src` is importable from notebooks/

import pandas as pd
from src.storage.mongo import get_collection

col = get_collection('closed_candles')

## 1. Total & per-symbol counts

In [2]:
print('Total documents:', col.count_documents({}))

pd.DataFrame(list(col.aggregate([
    {'$group': {
        '_id': '$symbol',
        'count': {'$sum': 1},
        'first': {'$min': '$kline_start_time'},
        'last':  {'$max': '$kline_start_time'},
    }},
])))

Total documents: 52


,_id,count,first,last
0,BTCUSDT,52,2026-06-16 09:34:00,2026-06-17 14:12:00


## 2. Latest 5 candles

In [3]:
docs = list(col.find().sort('kline_start_time', -1).limit(5))
pd.DataFrame(docs)

,_id,kline_start_time,symbol,close,high,interval,is_candle_closed,kline_close_time,low,number_of_trades,open,volume
0,6a32ab6be4101687c1f02212,2026-06-17 14:12:00,BTCUSDT,65327.76,65349.98,1m,True,2026-06-17 14:12:59.999,65268.89,3611,65277.77,15.10922
1,6a32ab2fe4101687c1f021af,2026-06-17 14:11:00,BTCUSDT,65277.76,65281.84,1m,True,2026-06-17 14:11:59.999,65205.99,2887,65206.00,5.36657
2,6a32aaf3e4101687c1f02150,2026-06-17 14:10:00,BTCUSDT,65206.00,65232.00,1m,True,2026-06-17 14:10:59.999,65194.32,2244,65212.00,5.65745
3,6a32aab7e4101687c1f020e7,2026-06-17 14:09:00,BTCUSDT,65212.00,65324.47,1m,True,2026-06-17 14:09:59.999,65204.00,3835,65324.47,12.27953
4,6a32aa7be4101687c1f02082,2026-06-17 14:08:00,BTCUSDT,65324.47,65351.62,1m,True,2026-06-17 14:08:59.999,65269.30,4754,65280.98,28.28903


## 3. Duplicate check
The unique index on `(symbol, kline_start_time)` should make this empty. Any rows here mean a candle was stored twice.

In [4]:
dups = list(col.aggregate([
    {'$group': {
        '_id': {'symbol': '$symbol', 'start': '$kline_start_time'},
        'count': {'$sum': 1},
    }},
    {'$match': {'count': {'$gt': 1}}},
]))

print(f'{len(dups)} duplicate (symbol, kline_start_time) keys found')
dups

0 duplicate (symbol, kline_start_time) keys found


[]

## 4. Gap check (1-min candles)
Consecutive `kline_start_time` should differ by 1 minute. Gaps mean either a missed message or the stream wasn't running.

In [5]:
SYMBOL = 'BTCUSDT'

df = pd.DataFrame(list(
    col.find({'symbol': SYMBOL}, {'_id': 0, 'kline_start_time': 1})
       .sort('kline_start_time', 1)
))

if df.empty:
    print('No candles yet — run stream_live.py first.')
else:
    df['gap'] = df['kline_start_time'].diff()
    gaps = df[df['gap'] > pd.Timedelta(minutes=1)]
    print(f'{len(df)} candles, {len(gaps)} gap(s)')
    gaps

15 candles, 0 gap(s)
